# 01 · Data validation across four systems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammedelsayed741996-blip/berlin-data-portfolio/blob/main/mobility/bus-network-demand-planning/notebooks/01_data_validation.ipynb)

A fictional intercity bus operator plans 22 lines between 32 cities in Germany and its neighbours.
Planning draws on four systems that are maintained by different people: the planning tool
(stops, lines, timetable), the partner CRM (bus partners and contracts), the operations log
(what actually ran, with which bus) and ticketing (bookings per trip).

All data is synthetic. City coordinates are public geography; partners are invented and marked
"(fictional)". Six problems were planted on purpose, and the checks below have to find all of them
before any KPI is trusted.

In [1]:
import os, sys, subprocess, warnings
from pathlib import Path

REPO = "https://github.com/mohammedelsayed741996-blip/berlin-data-portfolio.git"
PROJECT = "mobility/bus-network-demand-planning"

if "google.colab" in sys.modules:
    if not Path("/content/berlin-data-portfolio").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/berlin-data-portfolio"], check=True)
    os.chdir(f"/content/berlin-data-portfolio/{PROJECT}")
else:
    here = Path.cwd()
    while not (here / "src" / "bus_network").exists() and here != here.parent:
        here = here.parent
    os.chdir(here)

sys.path.insert(0, str(Path.cwd() / "src"))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)
from IPython.display import Image, display
print("Project folder:", Path.cwd().name)

Project folder: bus-network-demand-planning


In [2]:
from bus_network import config as C, data, validation as V, analysis as A, planning as P, reporting as R, db

if not any(Path("data").glob("*.csv*")):
    data.save(data.generate(), Path("data"))
tables = data.load(Path("data"))
LAST_12M = ("2025-10-01", "2026-09-30")
print({name: len(df) for name, df in tables.items()})

{'bookings': 80380, 'line_stops': 73, 'od_monthly': 6138, 'operations': 80180, 'partners': 9, 'stops': 33, 'timetable': 80, 'truth_demand': 80320}


## The tables

Every trip has an id such as `L010A-20260301-0800`: line, direction, date and departure time.
That id is the only key shared by ticketing and the operations log.

In [3]:
display(tables["timetable"].head(6))
display(tables["partners"])

,line,direction,dep_time,bus_type_planned,partner_id,origin,destination
0,L010,A,08:00,double,P07,Berlin,Munich
1,L010,A,10:00,double,P07,Berlin,Munich
2,L010,A,17:30,double,P07,Berlin,Munich
3,L010,B,08:00,double,P07,Munich,Berlin
4,L010,B,10:00,double,P07,Munich,Berlin
5,L010,B,17:30,double,P07,Munich,Berlin


,partner_id,partner_name,home_depot,contract_status,contract_end,fleet_single,fleet_double,account_owner
0,P01,Nordwind Reisen GmbH (fictional),Hamburg,active,2027-12-31,16,8,Partner Manager North
1,P02,Elbtal Bus GmbH (fictional),Dresden,active,2027-12-31,13,4,Partner Manager North
2,P03,Alpenlinie GmbH (fictional),Munich,active,2027-12-31,12,8,Partner Manager North
3,P04,Rheinland Touristik GmbH (fictional),Cologne,active,2027-12-31,8,7,Partner Manager North
4,P05,Ostsee Express GmbH (fictional),Rostock,active,2027-12-31,16,7,Partner Manager North
5,P06,Mainfranken Mobil GmbH (fictional),Nuremberg,active,2027-12-31,13,5,Partner Manager South
6,P07,Weichsel Coach GmbH (fictional),Berlin,expired,2026-06-30,9,9,Partner Manager South
7,P08,Schwarzwald Reisen GmbH (fictional),Freiburg,active,2027-12-31,14,5,Partner Manager South
8,P09,Hansa Fernbus GmbH (fictional),Bremen,active,2027-12-31,11,3,Partner Manager South


In [4]:
display(tables["operations"].head(3))
display(tables["bookings"].head(3))

,trip_id,date,line,direction,dep_time,partner_id,bus_type,seats,status,delay_min
0,L010A-20240101-0800,2024-01-01,L010,A,08:00,P07,double,79,operated,16.0
1,L010B-20240101-0800,2024-01-01,L010,B,08:00,P07,double,79,operated,6.0
2,L011A-20240101-0800,2024-01-01,L011,A,08:00,P02,double,79,operated,7.0


,trip_id,pax,pax_km,peak_load,peak_segment,revenue_eur,sold_out
0,L010A-20240101-0800,59,16331.0,43,Berlin-Leipzig,1108.65,False
1,L010B-20240101-0800,52,17187.8,37,Leipzig-Berlin,1130.39,False
2,L011A-20240101-0800,64,20147.3,39,Dresden-Prague,1339.26,False


## Eight checks, run before anything else

Each check returns the rows that fail it, plus an owner and an action, so the result can go straight
into an issue log rather than an email thread.

In [5]:
checks = V.run_all(tables)
issues = V.issue_log(checks)
issues[["check_id", "check", "status", "issues", "owner"]]

,check_id,check,status,issues,owner
0,V01,Stop coordinates lie in Europe,FAIL,1,Network data
1,V02,No duplicate stops (two records within 1 km),FAIL,1,Network data
2,V03,Every trip with passengers is in the operations log,FAIL,140,Operations / partner management
3,V04,Every booking belongs to a timetabled departure,FAIL,60,Ticketing / data
4,V05,Bus used matches the bus sold in ticketing,FAIL,1,Network planning / partner management
5,V06,Partners operate only under a valid contract,FAIL,1,Partner management
6,V07,Cancelled trips carry no passengers,PASS,0,Operations
7,V08,Trip bookings agree with origin-destination totals,FAIL,55,Ticketing / data


## What each failure means

The coordinates of one stop have latitude and longitude swapped, which puts Regensburg in the
Gulf of Aden on any map. Munich exists twice under two names 0.3 km apart, so its passengers are
split across two stop records. 140 trips sold tickets but are missing from the operations log, and
partner payments are based on that log.

In [6]:
for c in checks[:4]:
    print(c.check_id, c.name)
    display(c.rows.head(3))

V01 Stop coordinates lie in Europe
V02 No duplicate stops (two records within 1 km)
V03 Every trip with passengers is in the operations log
V04 Every booking belongs to a timetabled departure


,stop_id,stop_name,lat,lon
23,S24,Regensburg central bus station,12.1,49.01


,stop_a,name_a,stop_b,name_b,distance_km,lines_using_b
0,S03,Munich central bus station,S99,Muenchen ZOB,0.31,L031


,trip_id,pax,revenue_eur
63313,L040B-20260302-0800,41,248.30
63335,L020B-20260302-1000,96,1216.90
63341,L010B-20260302-1730,63,1527.51


,trip_id,pax,revenue_eur
80320,L012B-20251109-0545,40,732.33
80321,L020A-20250131-0545,113,1524.49
80322,L044A-20250608-0545,87,1288.51


The most expensive finding: since April 2026 the partner on line L017 has run single-deckers
(49 seats) while ticketing still sells a double-decker (79 seats). Most of those trips sold more
seats than the bus had. The SQL version of the check gives the same answer from the raw tables.

In [7]:
con = db.connect(tables)
display(db.run(con, Path("sql/02_capacity_mismatch.sql")))
display(db.run(con, Path("sql/03_expired_contracts.sql")))

,line,trips,overbooked_trips,max_seats_over,first_date
0,L017,362,314,30,2026-04-01


,partner_id,partner_name,contract_end,trips_after_end,first_trip,last_trip
0,P07,Weichsel Coach GmbH (fictional),2026-06-30,1279,2026-07-01,2026-09-30


## The clean trip table

Cancelled trips and bookings for departures that are not in the timetable are removed. Trips that
are missing from the log are kept, because the passengers did travel. Every later notebook works on
this table.

In [8]:
trips = V.clean_trips(tables)
print(f"{len(trips):,} trips from {trips['date'].min()} to {trips['date'].max()}")
trips.head(3)

79,656 trips from 2024-01-01 to 2026-09-30


,trip_id,pax,pax_km,peak_load,peak_segment,revenue_eur,sold_out,line,direction,date,dep_time,bus_type_planned,partner_id,origin,destination,status,seats,delay_min,line_km,seat_km,bus_cost_eur,month
0,L010A-20240101-0800,59,16331.0,43,Berlin-Leipzig,1108.65,False,L010,A,2024-01-01,08:00,double,P07,Berlin,Munich,operated,79.0,16.0,644.6,50923.4,999.130,2024-01
1,L010B-20240101-0800,52,17187.8,37,Leipzig-Berlin,1130.39,False,L010,B,2024-01-01,08:00,double,P07,Munich,Berlin,operated,79.0,6.0,644.6,50923.4,999.130,2024-01
2,L011A-20240101-0800,64,20147.3,39,Dresden-Prague,1339.26,False,L011,A,2024-01-01,08:00,double,P02,Berlin,Vienna,operated,79.0,7.0,652.3,51531.7,1011.065,2024-01
